In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import numpy as np
from scipy.stats import truncnorm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import image as img
from sklearn.model_selection import train_test_split
%matplotlib notebook

双卷积模块，每一个编码器和解码器通过双卷积模块进行

In [ ]:
class DoubleConv(nn.Module):
    """
    DoubleConv模块包含两个连续的卷积层，每个卷积层后接BatchNorm和ReLU激活函数。
    主要用于提取特征，同时保证特征尺寸不变
    """
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            # 第一个卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            # 第二个卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

U-net网络，由若干次编码、解码构成，这里取4次

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        """
        初始化UNet模型。
        :param in_channels: 输入通道数
        :param out_channels: 输出通道数
        """
        super(UNet, self).__init__()

        # 编码器部分（下采样路径）
        self.encoder1 = DoubleConv(in_channels, 32)  # 第一层编码器
        self.encoder2 = DoubleConv(32, 64)         # 第二层编码器
        self.encoder3 = DoubleConv(64, 128)       # 第三层编码器
        self.encoder4 = DoubleConv(128, 256)       # 第四层编码器
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # 最大池化层用于下采样

        # 瓶颈层
        self.bottleneck = DoubleConv(256, 512)  # 最底部特征提取层

        # 解码器部分（上采样路径）
        self.upconv4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)  # 上采样操作
        self.decoder4 = DoubleConv(512, 256)  # 解码器与跳跃连接结合特征

        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)  # 上采样操作
        self.decoder3 = DoubleConv(256, 128)  # 解码器与跳跃连接结合特征

        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)  # 上采样操作
        self.decoder2 = DoubleConv(128, 64)  # 解码器与跳跃连接结合特征

        self.upconv1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)  # 上采样操作
        self.decoder1 = DoubleConv(64, 32)  # 解码器与跳跃连接结合特征

        # 输出层，将特征图映射为分割结果
        self.final_conv = nn.Conv2d(32, out_channels, kernel_size=1)  # 输出卷积

    def forward(self, x):
        """
        定义UNet的前向传播。
        :param x: 输入图像
        :return: 分割结果张量
        """
        # 编码器路径：提取特征并逐步减少空间分辨率
        enc1 = self.encoder1(x)  # 第一层编码器
        enc2 = self.encoder2(self.pool(enc1))  # 第二层编码器
        enc3 = self.encoder3(self.pool(enc2))  # 第三层编码器
        enc4 = self.encoder4(self.pool(enc3))  # 第四层编码器

        # 瓶颈层，连接到解码器
        bottleneck = self.bottleneck(self.pool(enc4))

        # 解码器路径：逐步恢复空间分辨率，并结合编码器的跳跃连接
        dec4 = self.upconv4(bottleneck)  # 上采样第4层
        dec4 = torch.cat((enc4, dec4), dim=1)  # 跳跃连接结合编码器第4层
        dec4 = self.decoder4(dec4)  # 解码第4层

        dec3 = self.upconv3(dec4)  # 上采样第3层
        dec3 = torch.cat((enc3, dec3), dim=1)  # 跳跃连接结合编码器第3层
        dec3 = self.decoder3(dec3)  # 解码第3层

        dec2 = self.upconv2(dec3)  # 上采样第2层
        dec2 = torch.cat((enc2, dec2), dim=1)  # 跳跃连接结合编码器第2层
        dec2 = self.decoder2(dec2)  # 解码第2层

        dec1 = self.upconv1(dec2)  # 上采样第1层
        dec1 = torch.cat((enc1, dec1), dim=1)  # 跳跃连接结合编码器第1层
        dec1 = self.decoder1(dec1)  # 解码第1层

        # 最终输出层：生成分割结果
        return self.final_conv(dec1)

构建数据集

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]
        return image, mask

模型训练逻辑

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()  # 将模型设置为训练模式，启用 dropout 和 batch normalization 等特性
    epoch_loss = 0  # 初始化累计损失

    for images, masks in dataloader:  # 遍历数据加载器中的每个批次
        images, masks = images.to(device), masks.to(device)  # 将数据转移到指定设备（CPU 或 GPU）

        # Forward pass
        outputs = model(images)  # 前向传播：通过模型计算输出
        loss = criterion(outputs, masks)  # 根据输出和真实标签计算损失

        # Backward pass
        optimizer.zero_grad()  # 清零梯度，防止梯度累积
        loss.backward()  # 反向传播：计算梯度
        optimizer.step()  # 优化器更新模型参数

        epoch_loss += loss.item()  # 累加当前批次的损失值

    return epoch_loss / len(dataloader)  # 返回平均损失值


def validate(model, dataloader, criterion, device):
    model.eval()  # 将模型设置为评估模式，禁用 dropout 和 batch normalization 的随机特性
    val_loss = 0  # 初始化累计损失

    with torch.no_grad():  # 禁用梯度计算，加速和节省显存
        for images, masks in dataloader:  # 遍历验证集的每个批次
            images, masks = images.to(device), masks.to(device)  # 数据转移到设备
            outputs = model(images)  # 前向传播：通过模型计算输出
            loss = criterion(outputs, masks)  # 计算损失
            val_loss += loss.item()  # 累加当前批次的损失值

    return val_loss / len(dataloader)  # 返回平均损失值

# 划分训练集、测试集、验证集

In [ ]:
def split_dataset(images, masks, train=0.8, validation=0.1, test=0.1, random_state=None):
    if train+validation+test != 1:
        return -1

    train_images, temp_images, train_masks, temp_masks = train_test_split(images, masks, test_size=1-train, random_state=random_state)
    val_images, test_images, val_masks, test_masks = train_test_split(temp_images, temp_masks, test_size=test/(validation+test), random_state=random_state)

    train_dataset = CustomDataset(train_images, train_masks)
    test_dataset = CustomDataset(test_images, test_masks)
    val_dataset = CustomDataset(val_images, val_masks)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    return train_loader, val_loader, test_loader

# 训练、测试模型及使用模型预测

In [ ]:
class UNetTest():
    def __init__(self, in_channels=1, out_channels=1):
        self.model = UNet(in_channels=in_channels, out_channels=out_channels).to('cuda') # 加载模型
        self.criterion = nn.BCEWithLogitsLoss()  # 损失函数
        self.optimizer = optim.Adam(self.model.parameters(), lr=1e-4) # 优化器

    def train_model(self, epochs, train_loader, val_loader, plot=True):
        best_val_loss = float("inf")
        train_losses = []
        val_losses = []

        for epoch in tqdm(range(epochs), desc="Epoch", total=epochs):
            train_loss = train_one_epoch(self.model, train_loader, self.optimizer, self.criterion, 'cuda')
            val_loss = validate(self.model, val_loader, self.criterion, 'cuda')

            train_losses.append(train_loss)
            val_losses.append(val_loss)

            # 保存最优模型
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(self.model.state_dict(), "./pytorch/best_unet_model.pth")

        if plot:
            fig, ax = plt.subplots()
            ax.clear()
            ax.plot(range(1, 51), train_losses, label='Train Loss', marker='o')
            ax.plot(range(1, 51), val_losses, label='Val Loss', marker='o')
            ax.legend()
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.grid()
            fig.savefig("./fig/loss.png", dpi=300)

    def test_model(self, test_loader, path="./pytorch/best_unet_model.pth"):
        self.model.load_state_dict(torch.load(path, weights_only=True))
        self.model.eval()

        test_loss = validate(self.model, test_loader, self.criterion, 'cuda')
        print(f"Test Loss: {test_loss:.4f}")

    def predict(self, image, path="./pytorch/best_unet_model.pth"):
        image_tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0).to('cuda')

        self.model.load_state_dict(torch.load(path, weights_only=True))
        self.model.eval()

        with torch.no_grad():
            output = self.model(image_tensor)
            predictions = torch.sigmoid(output) > 0.5

        output_image = predictions.squeeze().cpu().numpy()
        note_image = np.copy(image)
        note_image[output_image] = np.nan

        return output_image, note_image